In [6]:
import pandas as pd
import numpy as np 
import warnings
warnings.filterwarnings("ignore")

Nota: Para este ejemplo asumiremos que los datos de entrenamiento son sobre los que trabajaremos

In [9]:
Y = np.array([[3], [1], [8], [3], [5]])
Y

array([[3],
       [1],
       [8],
       [3],
       [5]])

In [11]:
X = np.array ([[1,3,5], [1,1,4],[1,5,6], [1,2,4], [1,4,6]])
X

array([[1, 3, 5],
       [1, 1, 4],
       [1, 5, 6],
       [1, 2, 4],
       [1, 4, 6]])

In [13]:
# Matriz X transpuesta
XT_X = np.matmul(np.matrix.transpose(X), X)
XT_X

array([[  5,  15,  25],
       [ 15,  55,  81],
       [ 25,  81, 129]])

In [15]:
# Inversa de la traspuesta
XT_X_inv = np.linalg.inv(XT_X)
XT_X_inv

array([[26.7,  4.5, -8. ],
       [ 4.5,  1. , -1.5],
       [-8. , -1.5,  2.5]])

In [17]:
XT_Y = np.matmul(np.matrix.transpose(X), Y)
XT_Y

array([[ 20],
       [ 76],
       [109]])

In [19]:
betas = np.matmul(XT_X_inv, XT_Y)
betas

array([[ 4. ],
       [ 2.5],
       [-1.5]])

In [21]:
# Calculo de los prnosticos para Y de acuerdo a los coeficientes de regresión
Y_pred = np.matmul(X, betas)
Y_pred

array([[4. ],
       [0.5],
       [7.5],
       [3. ],
       [5. ]])

In [23]:
# Calculo de residuales
Resid = Y - Y_pred
Resid.round(2)

array([[-1. ],
       [ 0.5],
       [ 0.5],
       [-0. ],
       [-0. ]])

In [25]:
# Calculo de la suma de residuales al cuadrado
RSS = float(np.matmul(np.matrix.transpose(Resid), Resid))
RSS

1.4999999999999991

In [27]:
# Calculo de la suma total de cuadrados
TSS = float(np.matmul(np.matrix.transpose(Y), Y) - len(Y)*(Y.mean()**2))
TSS

28.0

In [29]:
# Calculo de coeficiente de determinación 
R_cuad = float(1- RSS/TSS)
R_cuad

0.9464285714285715

In [31]:
# Calculo de coeficiente de determianción ajustado
RSqAj = float(1-(RSS/ (X.shape[0]- X.shape[1])) / (TSS / (X.shape[0]-1)))
RSqAj

0.8928571428571429

In [33]:
# Calculo de la varianza del error de regresión
s_cuad = RSS / (len(Y) - X.shape[1])
s_cuad

0.7499999999999996

In [35]:
# Desviación estandar del error de regresión
import math
s = math.sqrt(s_cuad)
s

0.8660254037844384

In [37]:
# Calculo de las t's estadisticas para cada coeficiente de regresión
result_t = []
for i in range(0, X.shape[1]):
    t = float(betas[i] / (s * math.sqrt(XT_X_inv[i][i])))
    result_t.append(t)
result_t

[0.893868697538675, 2.8867513459481344, -1.0954451150103264]

# Criterio 1:

In [42]:
# Obtener el valor critico de la t de Student de tablas
import scipy.stats

grados_libertad = len(Y) - X.shape[1]
# La t_critica se obtendrá de un nivel de confianza del 95% (Alfa = 5%)
t_critico = abs(scipy.stats.t.ppf(q = 0.025, df = grados_libertad))
t_critico

4.302652729749464

In [44]:
for i in range(0, X.shape[1]):
    if (abs(result_t[i]) > t_critico):
        print("Beta", i ,"es significativa") # Aquí se rechaza H0
    else:
        print("Beta", i ,"NO es significativa") # Aquí NO se rechaza H0


Beta 0 NO es significativa
Beta 1 NO es significativa
Beta 2 NO es significativa


# Criterio 2:

In [49]:
# Calcxulo de valores p 
for i in range(0, X.shape[1]):
    print("Valor p de Beta", i, ":", scipy.stats.t.sf(abs(result_t[i]), df = grados_libertad) * 2)

Valor p de Beta 0 : 0.46571598260852526
Valor p de Beta 1 : 0.10197348986612516
Valor p de Beta 2 : 0.38762756430420753


Si manejamos un nivel alfa del 5%, un ninguno de lso casos el valor p es menor al 5%, por lo que no podemos rechazar H0

# Criterio 3:

In [53]:
# Calculo de intervalos de confianza del 95% para el verdadero valor del coeficiente de cada Beta
for i in range(0, X.shape[1]):
    print("El avlor de Beta", i, "se encuentra entre", float(betas[i]) - t_critico * s * math.sqrt(XT_X_inv[i][i]),
         "y", float(betas[i]) + t_critico * s * math.sqrt(XT_X_inv[i][i]))

El avlor de Beta 0 se encuentra entre -15.25407049870823 y 23.25407049870841
El avlor de Beta 1 se encuentra entre -1.2262065676254923 y 6.226206567625505
El avlor de Beta 2 se encuentra entre -7.391649892987392 y 4.391649892987408


Conlusión: Ninguna de las variable regresoras (independientes) es significativa de cero

In [62]:
# Comparación de resultados contra reporrte automatizado
import statsmodels.api as sm

regressor = sm.OLS(Y, X).fit()
print(regressor.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.946
Model:                            OLS   Adj. R-squared:                  0.893
Method:                 Least Squares   F-statistic:                     17.67
Date:                Sun, 22 Mar 2026   Prob (F-statistic):             0.0536
Time:                        12:52:40   Log-Likelihood:                -4.0848
No. Observations:                   5   AIC:                             14.17
Df Residuals:                       2   BIC:                             13.00
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          4.0000      4.475      0.894      0.4

C:\Users\AlanHDLR\anaconda3\Lib\site-packages\statsmodels\stats\stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 5 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [64]:
data = pd.DataFrame(X)
data2 = data.iloc[:,1:3]
data2.corr()

,1,2
1,1.000000,0.948683
2,0.948683,1.000000


In [66]:
X_Nueva = np.delete(X, 2, 1)
X_Nueva

array([[1, 3],
       [1, 1],
       [1, 5],
       [1, 2],
       [1, 4]])

In [68]:
regressor = sm.OLS(Y, X_Nueva).fit()
print(regressor.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.914
Model:                            OLS   Adj. R-squared:                  0.886
Method:                 Least Squares   F-statistic:                     32.00
Date:                Sun, 22 Mar 2026   Prob (F-statistic):             0.0109
Time:                        12:58:35   Log-Likelihood:                -5.2598
No. Observations:                   5   AIC:                             14.52
Df Residuals:                       3   BIC:                             13.74
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.8000      0.938     -0.853      0.4

C:\Users\AlanHDLR\anaconda3\Lib\site-packages\statsmodels\stats\stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 5 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "
